# 02 · Clean eval (best 체크포인트, 500ep + action 기록)
**libero_10**(메인 sim) best 체크포인트 500ep 재평가 + action .pt(→jerk). SELECT로 ckpt 선택.
아래 '(선택) horizon' 셀 = 같은 모델을 LIBERO suite(spatial→goal→10)로 평가 → sim horizon 곡선.
⚠️ LIBERO 시뮬 필요(먼저 `90_install_libero`). eval은 한 머신에서 GPU 0-3 사용.

In [ ]:
import sys, os
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))
import common_final as cf
print('FINAL_TAGS:', cf.FINAL_TAGS, '| seeds:', cf.FINAL_SEEDS)
print('OUTPUT_BASE:', cf.OUTPUT_BASE)

In [ ]:
SEEDS = [0, 1, 2, 3]       # 있는 seed만
TASK = cf.PRIMARY_SIM      # 'libero_10'
SELECT = 'best'            # best-SR ckpt (또는 'last', 정수 step)
N_EP = 500
labeled = []
for t in cf.FINAL_TAGS:
    for s in SEEDS:
        try:
            cmd = cf.make_eval_cmd(t, seed=s, task=TASK, gpu_id=len(labeled) % 4,
                                   n_episodes=N_EP, select=SELECT)
            labeled.append((cf.run_label(t, s, TASK), cmd))
        except FileNotFoundError:
            print('skip (ckpt 없음):', t, s)
print(len(labeled), 'eval 잡')
cf.launch_cmds_live(labeled, log_tag='eval')

In [ ]:
# (선택) sim horizon 곡선: libero_10로 학습한 모델을 suite별 평가 (spatial→goal→10)
# ⚠️ LIBERO 시뮬 필요. run_perturb_eval가 --env.type=libero 를 받아야 함(안 되면 lerobot_eval로 대체).
HZ_SEEDS = [0, 1]
hz = []
for t in cf.FINAL_TAGS:
    for s in HZ_SEEDS:
        for suite in cf.LIBERO_HORIZON:   # ['libero_spatial','libero_goal','libero_10']
            try:
                cmd = cf.libero_suite_eval_cmd(t, s, suite, gpu_id=len(hz) % 4, n_episodes=200)
                hz.append((f'{t}/seed{s}/{suite}', cmd))
            except FileNotFoundError:
                pass
print(len(hz), 'horizon eval 잡')
cf.launch_cmds_live(hz, log_tag='eval_hz')

In [ ]:
# eval SR 상태
for t in cf.FINAL_TAGS:
    for s in SEEDS:
        st = cf.get_eval_status(t, s, TASK)
        sr = f"{st['sr']*100:.1f}%" if st['sr'] is not None else '-'
        print(f'{t:12} seed{s}  SR={sr}  done={st["done"]}')